In [ ]:
import os
import sys

project_root = '/home/jovyan/project_10x'

sys.path.append(os.path.join(project_root, 'src'))

from utils import *
from sentiment_extraction_MB_10k import *

import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
def clean_mem():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    time.sleep(0.1)

clean_mem()

In [ ]:
sys.version

# Model - ModernBERT

In [ ]:
model_id = "/home/jovyan/models-2/ModernBERT-large"

tokenizer = AutoTokenizer.from_pretrained(model_id,
#                                           max_length=length,
                                          padding=True,
                                          truncation=True,
                                          cache_dir="hf_cache")
if not tokenizer.pad_token:
    print("Adding pad token")
    tokenizer.pad_token = tokenizer.eos_token
    
tokenizer.padding_side = "left"

model = AutoModelForMaskedLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, cache_dir="hf_cache"
).to(device)

config = AutoConfig.from_pretrained(model_id)
length = config.max_position_embeddings
length

In [ ]:
model.name = 'Llama3_1'

In [ ]:
def get_prompt(item_text: str, ending: str) -> str:
    return item_text + "\n\n" + ending 

## Whole dataset

In [ ]:
report_path = '/home/jovyan/datavol-2/'

reports = pd.read_csv(os.path.join(report_path, '10k_filtered.tsv.gz'), sep='\t')
reports = reports[['CIK', 'FILING_DATE', 'ACC_NUM']]
reports['CIK'] = reports['CIK'].astype(str)
reports['FILING_DATE'] = reports['FILING_DATE'].astype(str)
reports['ACC_NUM'] = reports['ACC_NUM'].astype(str)
# reports

df = pd.read_sas('/home/jovyan/datavol-2/finaldata_10k.sas7bdat')

df['ACC_NUM'] = df['filename'].astype(str).str.split('_').str[1]
df['CIK'] = df['filename'].astype(str).str.split('_').str[0]
df['FILING_DATE'] = df['date_filed'].astype(str).str.replace('-', '')
# reports = df[df['FILING_DATE'].astype(str).str[:4] < '2013'].reset_index(drop=True)
df = df[['CIK', 'FILING_DATE', 'ACC_NUM']]
df['CIK'] = df['CIK'].astype(str)
df['FILING_DATE'] = df['FILING_DATE'].astype(str)
df['ACC_NUM'] = df['ACC_NUM'].astype(str)
# reports

missing = pd.read_csv('/home/jovyan/datavol-2/missing_item7.tsv.gz', sep='\t')

missing['ACC_NUM'] = missing['name'].astype(str).str.split('_').str[-1]
missing['CIK'] = missing['name'].astype(str).str.split('_').str[-2]
missing['FILING_DATE'] = missing['name'].astype(str).str.split('_').str[0]
missing = missing[['CIK', 'FILING_DATE', 'ACC_NUM']]

In [ ]:
res_df = pd.concat([reports, df, missing], axis=0)
res_df = res_df.drop_duplicates().reset_index(drop=True)

res_df.shape, res_df.drop_duplicates().shape

In [ ]:
res_df.info()

In [ ]:
res_df

In [ ]:
res_df.to_csv('/home/jovyan/datavol-2/df_whole_sample.csv', index=True)

In [ ]:
items_path = '/home/jovyan/datavol-2/items/item7_files'

dataset = Dataset10x(res_df, items_path)
len(dataset)
# 109921

In [ ]:
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer=tokenizer,
                                                                     chunk_size=length,
                                                                     chunk_overlap=240)

## Growth Sentiment Extraction

### In one word, we are [MASK] regarding the future growth of our company

In [ ]:
ending = "In one word, we are [MASK] regarding the future growth of our company."

In [ ]:
idx = 11_900

_, item_dict = dataset[idx]
date = reports.iloc[idx].FILING_DATE
item_text = item_dict.get('item7')[:1000]

In [ ]:
prompt = get_prompt(item_text, ending)
probs = get_model_output(prompt, model, device)
probs

In [ ]:
clean_mem()

In [ ]:
@dataclass(frozen=True)
class Prompt_Strategy:
    name: str
    verbalizer: dict
    prompt: Callable
    top_p: float = 0.99

sentiment_verb = {
    "positive": set(['optimistic', 'confident', 'positive', 'encouraged', 'excited',
                     'enthusiastic', 'hopeful', 'pleased', 'encouraging', 'ambitious',
                     'favorable', 'assured', 'strong', 'good', 'excellent', 'outstanding', 
                     'healthy', 'awesome', 'great', 'fantastic', 'stable',
                     'perfect', 'solid', 'profitable', 'impressive', 'reliable',
                     'thriving', 'optimistic', 'sustainable',
                     'exceptional', 'promising', 'bright', 'attractive',
                     'comfortable', 'satisfied', 'relaxed', 'satisfied', 'comfortable',
                     'ambitious', 'determined', 'bullish', 'upbeat', 'convinced', 
                     'certain', 'reassured',
                     ]),
    
    "negative": set(['cautious', 'concerned', 'pessimistic', 'negative',
                     'uncomfortable', 'uncertain', 'unsure',
                     'skeptical', 'worried', 'worrying', 'discouraged', 
                     'confused', 'doubtful', 'unsatisfied', 'disappointed',
                     'bad', 'poor', 'terrible', 'risky', 'weak', 'dependent',
                     'unstable', 'unhealthy', 'questionable', 'suffering', 'stressed',
                     'unsustainable', 'awful', 'vulnerable',
                     'mediocre', 'horrible', 'precarious', 'declining', 'worsening',
                     'difficult', 'limited', 'challenged', 'disappointed', 'discouraged',
                     'frustrated', 'dissatisfied', 'worried', 'anxious', 'nervous', 'uneasy',
                     'doubtful', 'skeptical', 'apprehensive',
                    ])
}

sentiment_strategy = Prompt_Strategy('sentiment', sentiment_verb, get_prompt)

len(sentiment_verb['positive']), len(sentiment_verb['negative'])

In [ ]:
q99 = 275568
q95 = 169981

In [ ]:
batch_size = 2

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=split_collator
)

len(dataloader)

In [ ]:
clean_mem()

In [ ]:
ending = "In one word, we are [MASK] regarding the future growth of our company."

In [ ]:
results = []

In [ ]:
stats_sent = gather_stats(sentiment_strategy, results=results, tokenizer=tokenizer, model=model,
                          data=dataloader, ending=ending, verbose=False, text_length=q99, 
                          save_path="/home/jovyan/datavol-2/sentiments/growth_sentiment_MB/",
                          save_interval=1000, resume=True, max_retries=3, device=device)

In [ ]:
clean_mem()

In [ ]:
time.sleep(60)
clean_mem()
torch.cuda.empty_cache()
time.sleep(60)

In [ ]:
stats_sent.to_csv('/home/jovyan/datavol-2/sentiments/growth_sentiment_MB/results_full_growth_MB.csv')

In [ ]:
stats_sent['polarity'].hist(bins=100)